<a href="https://colab.research.google.com/github/Romulus09/TCC/blob/main/TCC_train_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
#!pip install datasets mlcroissant
import numpy as np
from datasets import load_dataset
from transformers import (
    RobertaTokenizerFast,
    RobertaForTokenClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
#import evaluate
import torch

In [8]:
dataset = load_dataset("rjac/kaggle-entity-annotated-corpus-ner-dataset")
#dataset = dataset.select(range(100))

# O dataset possui colunas: tokens, ner_tags
# Transformando o problema em binário (0 = O, 1 = entidade)
def convert_to_binary(example):
    example["binary_labels"] = [0 if tag == 0 else 1 for tag in example["ner_tags"]]
    return example

dataset = dataset.map(convert_to_binary)


In [9]:
# =======================================================
# 2. Tokenizer do RoBERTa
# =======================================================
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base",add_prefix_space=True)

def tokenize_and_align_labels(batch):
    tokenized = tokenizer(
        batch["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128
    )
    labels = []

    for i, words in enumerate(batch["tokens"]):
        word_ids = tokenized.word_ids(batch_index=i)
        example_labels = batch["binary_labels"][i]
        aligned_labels = []

        prev_word = None
        for word_id in word_ids:
            if word_id is None:
                aligned_labels.append(-100)
            else:
                aligned_labels.append(example_labels[word_id])
        labels.append(aligned_labels)

    tokenized["labels"] = labels
    return tokenized

encoded = dataset.map(tokenize_and_align_labels, batched=True)

In [10]:
# =======================================================
# 3. Função de métrica AUC-ROC binária
# =======================================================
def compute_auc_roc(eval_pred):
    logits, labels = eval_pred
    logits = torch.tensor(logits)
    labels = torch.tensor(labels)

    # Extrair logits dos tokens válidos (labels != -100)
    mask = labels != -100
    logits = logits[mask]
    labels = labels[mask]

    # Converter logits → probabilidades da classe 1
    probs = torch.softmax(logits, dim=-1)[:, 1].numpy()
    y_true = labels.numpy()

    auc = roc_auc_score(y_true, probs)
    return {"auc_roc": auc}

In [11]:
from transformers import (
    RobertaTokenizerFast,
    RobertaForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification # Added import
)

# =======================================================
# 4. K-FOLD CROSS VALIDATION (k=5)
# =======================================================
k = 5
kf = KFold(n_splits=k, shuffle=True, random_state=42)

X = np.arange(len(encoded["train"]))  # índices da base
results = []

# Initialize the data collator for token classification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer) # Added data collator

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    print(f"\n======== FOLD {fold+1}/{k} ========")

    train_split = encoded["train"].select(train_idx)
    eval_split = encoded["train"].select(test_idx)

    model = RobertaForTokenClassification.from_pretrained(
        "roberta-base",
        num_labels=2  # binário: entidade vs não entidade
    )

    training_args = TrainingArguments(
        output_dir=f"./results_fold_{fold}",
        learning_rate=3e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="steps",
        logging_steps=50,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_split,
        eval_dataset=eval_split,
        #tokenizer=tokenizer, # Keep tokenizer here for default processing class for eval
        data_collator=data_collator, # Pass the data collator
        compute_metrics=compute_auc_roc,
    )

    trainer.train()
    metrics = trainer.evaluate()

    print(f"AUC-ROC do fold {fold+1}: {metrics['eval_auc_roc']:.4f}")
    results.append(metrics["eval_auc_roc"])


======== FOLD 1/5 ========


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 